# Impetus — Energy-Based Reasoning for Open LLMs
## Kaggle Notebook | Baseline Benchmarking

**Strategy:** Math + Logic first (GSM8K, ARC, BBH)

**Pipeline:** Prompt → N candidates → Energy scoring → Best answer

---
### Instructions
1. Settings → **Accelerator → GPU T4 x1** (if P100 is assigned, notebook handles it)
2. **Internet** must be ON (Settings → Internet)
3. Run all cells
4. Results saved as CSV in `/kaggle/working/results/`
5. Download CSV or submit to GitHub

In [ ]:
import sys, os, subprocess
from pathlib import Path

# Detect GPU
gpu_info = subprocess.run(
    ["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
    capture_output=True, text=True
).stdout.strip().lower()
print(f"Detected GPU: {gpu_info}")

# P100 (sm_60) is incompatible with PyTorch >= 2.1; install older torch
if "p100" in gpu_info:
    print("P100 detected - installing PyTorch 2.0.1 (compatible with sm_60)")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
        "torch==2.0.1", "torchvision==0.15.2",
        "--index-url", "https://download.pytorch.org/whl/cu117"])
else:
    print("Modern GPU detected - installing latest PyTorch")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
        "torch>=2.1.0", "torchvision>=0.16.0",
        "--index-url", "https://download.pytorch.org/whl/cu118"])

# Install rest of dependencies
DEPS = [
    "transformers>=4.36.0",
    "accelerate>=0.25.0",
    "datasets>=2.16.0",
    "evaluate>=0.4.1",
    "sentencepiece>=0.1.99",
    "scipy", "pandas", "tqdm",
]
for dep in DEPS:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", dep])

print("Dependencies installed.")

In [ ]:
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    print(f"Device: {gpu_name}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    # P100 doesn't support bfloat16; T4/V100/A100 do
    DTYPE = torch.float16 if "P100" in gpu_name else torch.bfloat16
    print(f"Using dtype: {DTYPE}")

In [ ]:
# Clone/pull project
REPO_URL = "https://github.com/EdhieBM/Impetus.git"
PROJECT_DIR = "/kaggle/working/Impetus"

if os.path.exists(PROJECT_DIR):
    subprocess.run(["git", "-C", PROJECT_DIR, "pull"])
else:
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, PROJECT_DIR])

os.chdir(PROJECT_DIR)
sys.path.insert(0, PROJECT_DIR)
print(f"Working in: {os.getcwd()}")
print(f"Files: {os.listdir('.')}")

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"

print(f"Loading {MODEL_NAME}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
tokenizer.padding_side = "left"
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=DTYPE,
    device_map="auto",
    trust_remote_code=True,
)
model.eval()
print("Model loaded successfully")

In [ ]:
# Phase 1: Baseline benchmark (GSM8K) with batched generation
from benchmarks.run_baseline import run_gsm8k, save_results

print("Running GSM8K baseline (50 samples, batch_size=8)...")
results = run_gsm8k(model, tokenizer, max_samples=50, batch_size=8)
path = save_results(results, MODEL_NAME, "results")
print(f"Saved: {path}")

In [ ]:
# Evaluate
from benchmarks.evaluate_results import evaluate_gsm8k

stats = evaluate_gsm8k(results)
print(f"GSM8K  | Acc: {stats['accuracy']:.2%}  ({stats['correct']}/{stats['total']})  | Latency: {stats['avg_latency_s']:.2f}s")

In [ ]:
# Phase 2: KONA verifier full benchmark on GSM8K
from benchmarks.run_reranker import run_reranker_gsm8k, save_results
from energy_module.scorers import MajorityVotingScorer

print("Running GSM8K reranker benchmark (50 samples, N=5 candidates, majority_voting)...")
scorer = MajorityVotingScorer()
results_reranker = run_reranker_gsm8k(
    model, tokenizer, scorer,
    max_samples=50, n_candidates=5, batch_size=8
)
path2 = save_results(results_reranker, MODEL_NAME, "majority_voting", "results")
print(f"Saved: {path2}")

In [ ]:
import pandas as pd

# Summarize reranker results
baseline_correct = sum(1 for r in results_reranker if r["baseline_correct"] == "True")
reranked_correct = sum(1 for r in results_reranker if r["reranked_correct"] == "True")
total = len(results_reranker)

print("\n" + "="*55)
print("FINAL COMPARISON: Baseline vs Reranker")
print("="*55)
print(f"Model: {MODEL_NAME}")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'}")
print(f"N candidates: 5 | Scorer: majority_voting\n")
print(f"Baseline (greedy):             {baseline_correct}/{total} = {baseline_correct/total:.2%}")
print(f"Reranked (majority voting):    {reranked_correct}/{total} = {reranked_correct/total:.2%}")
delta = (reranked_correct - baseline_correct) / total
print(f"Delta: {delta:+.2%}")
if delta > 0:
    print("> ENERGY-BASED RERANKING IMPROVES ACCURACY!")
else:
    print("> Reranking did not improve accuracy (need better scorer)")
print("="*55)